# One2lvOS on Colab · Ubuntu 26 terminal · WebOS desktop · Lumenis

<a href="https://colab.research.google.com/github/one2lv-com/One2lvos/blob/main/One2lvOS_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Run the cells in order (or **Runtime → Run all**). At the end you get one public link that opens:

- **one2lvos-desktop.html** as the desktop, served by the Lumenis reactor
- **Lumenis Chat** built into the desktop (talks to the reactor over `/ws`), plus the full **Lumenis Console** (`lumenis-chat.html`) as a window
- **Ubuntu 26 Terminal**: a real shell in the browser (ttyd over a WebSocket) where `sudo apt update`, `sudo apt upgrade`, `sudo su` and `su -` all work

**How Ubuntu 26 works here.** Colab's own image is an older Ubuntu, and upgrading it in place would replace the libraries the notebook itself runs on and break the runtime. So this notebook downloads the official Ubuntu 26 base image, points apt at the Ubuntu 26 mirrors, and runs it as a full root filesystem with `chroot`. Everything inside the terminal is real Ubuntu 26 (apt, sudo, su, the packages); only the kernel is Colab's.

**Secrets.** Nothing is stored in this notebook. Add these in Colab's 🔑 **Secrets** panel (and turn on notebook access), or type them when asked:

| Secret | Needed | Used for |
|---|---|---|
| `O2_DESKTOP_PASSWORD` | yes | the login page in front of the public link, and the root password for `su -` |
| `NVIDIA_API_KEY` | for chat | Lumenis's replies |

In [ ]:
#@title 1 · Settings and secrets
UBUNTU_RELEASE = "26.04.1 LTS (resolute)"  #@param ["26.04.1 LTS (resolute)", "26.10 (stonking)"]
APT_MIRROR = "http://archive.ubuntu.com/ubuntu/"  #@param {type:"string"}
RUN_FULL_UPGRADE = True  #@param {type:"boolean"}
REPO_BRANCH = "main"  #@param {type:"string"}

import os, sys, getpass

# Paths and ports. Everything listens on 127.0.0.1; only the tunnel in step 5 is public.
BASE       = os.environ.get("O2_BASE", "/content")
REPO_DIR   = os.environ.get("O2_REPO_DIR", f"{BASE}/One2lvos")
UBU_ROOT   = os.environ.get("O2_ROOT", "/opt/ubuntu26")
BIN_DIR    = os.environ.get("O2_BIN_DIR", "/usr/local/bin")
LOG_DIR    = os.environ.get("O2_LOG_DIR", f"{BASE}/o2-logs")
REACTOR_PORT = int(os.environ.get("O2_REACTOR_PORT", "8000"))
TTYD_PORT    = int(os.environ.get("O2_TTYD_PORT", "7681"))
GATEWAY_PORT = int(os.environ.get("O2_GATEWAY_PORT", "8090"))
UBU_VERSION  = UBUNTU_RELEASE.split()[0]          # 26.04.1 or 26.10
os.makedirs(LOG_DIR, exist_ok=True)

def _secret(name, prompt, required):
    val = os.environ.get(name, "")
    if not val:
        try:
            from google.colab import userdata
            val = userdata.get(name) or ""
        except Exception:
            val = ""
    if not val and (required or "google.colab" in sys.modules):
        val = getpass.getpass(prompt)
    return val.strip()

DESKTOP_PASSWORD = _secret("O2_DESKTOP_PASSWORD", "Desktop password (for the login page and `su -`): ", True)
if len(DESKTOP_PASSWORD) < 8:
    raise SystemExit("Pick a desktop password of at least 8 characters (Colab secret O2_DESKTOP_PASSWORD).")
NVIDIA_API_KEY = _secret("NVIDIA_API_KEY", "NVIDIA API key for Lumenis (Enter to skip): ", False)

# Kept in this runtime's environment only, never written into the notebook.
os.environ["O2_DESKTOP_PASSWORD"] = DESKTOP_PASSWORD
if NVIDIA_API_KEY:
    os.environ["NVIDIA_API_KEY"] = NVIDIA_API_KEY
print(f"Ubuntu {UBU_VERSION} · mirror {APT_MIRROR} · root {UBU_ROOT}")
print("Desktop password: set · NVIDIA key:", "set" if NVIDIA_API_KEY else "not set (Lumenis chat will not answer)")

In [ ]:
#@title 2 · Get One2lvOS and the host tools (ttyd, cloudflared, Python packages)
import subprocess, sys, platform, shutil, urllib.request

def sh(cmd, check=True, quiet=False, env=None):
    """Run a shell command and stream its output."""
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         env=env or os.environ.copy())
    out = []
    for line in p.stdout:
        out.append(line)
        if not quiet: print(line, end="")
    p.wait()
    if check and p.returncode:
        if quiet: print("".join(out[-40:]))
        raise SystemExit(f"Command failed ({p.returncode}): {cmd.split()[0]}")
    return "".join(out)

print("Colab host:", sh(". /etc/os-release && echo $PRETTY_NAME", quiet=True).strip(), "· kernel", platform.release())

# One2lvOS repo (public). Pull if it is already here.
if os.path.isdir(f"{REPO_DIR}/.git"):
    sh(f"git -C {REPO_DIR} fetch -q origin {REPO_BRANCH} && git -C {REPO_DIR} checkout -q {REPO_BRANCH} && git -C {REPO_DIR} reset -q --hard origin/{REPO_BRANCH}")
elif not os.path.isdir(REPO_DIR):
    sh(f"git clone -q --depth 1 -b {REPO_BRANCH} https://github.com/one2lv-com/One2lvos.git {REPO_DIR}")
sh(f"git -C {REPO_DIR} log -1 --format='One2lvOS %h · %s' 2>/dev/null || true")
for f in ("one2lvos-desktop.html", "static/lumenis-chat.html", "lumenis-main.py", "colab/gateway.py", "colab/ubuntu26-root.sh"):
    assert os.path.exists(f"{REPO_DIR}/{f}"), f"missing {f} in the repo"

def fetch_bin(name, url):
    dest = f"{BIN_DIR}/{name}"
    if not (os.path.exists(dest) and os.access(dest, os.X_OK)):
        os.makedirs(BIN_DIR, exist_ok=True)
        urllib.request.urlretrieve(url, dest + ".part"); os.chmod(dest + ".part", 0o755); os.replace(dest + ".part", dest)
    print(f"{name}: " + sh(f"{dest} --version 2>&1 | head -1", quiet=True).strip())
    return dest

TTYD = fetch_bin("ttyd", "https://github.com/tsl0922/ttyd/releases/latest/download/ttyd.x86_64")
CLOUDFLARED = fetch_bin("cloudflared", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64")

# What the Lumenis reactor needs (Colab already ships most of it).
if not os.environ.get("O2_SKIP_PIP"):
    sh(f"{sys.executable} -m pip install -q fastapi 'uvicorn[standard]' websockets httpx aiohttp python-dotenv "
       "pydantic requests numpy pillow openai", quiet=True)
sh(f"{sys.executable} -c 'import fastapi, uvicorn, aiohttp, httpx; print(\"Python packages ready\")'")

In [ ]:
#@title 3 · Build the Ubuntu 26 root: Ubuntu 26 mirrors, apt update, apt upgrade, sudo, su
# First run takes a few minutes (download ~30 MB, then apt). Re-running refreshes it.
env = os.environ.copy()
env.update(O2_ROOT=UBU_ROOT, O2_UBUNTU=UBU_VERSION, O2_MIRROR=APT_MIRROR, O2_USER="one2lv",
           O2_ROOT_PASSWORD=DESKTOP_PASSWORD, O2_UPGRADE="1" if RUN_FULL_UPGRADE else "0")
sh(f"bash {REPO_DIR}/colab/ubuntu26-root.sh", env=env)
UBU_MODE = open(f"{UBU_ROOT}/.o2-mode").read().strip()
print("\nInside the root:", [l.split("=",1)[1].strip().strip('"') for l in open(f"{UBU_ROOT}/etc/os-release") if l.startswith("PRETTY_NAME")][0], "via", UBU_MODE)

In [ ]:
#@title 4 · Start the Lumenis reactor, the terminal and the gateway
import time, json, signal, urllib.request, urllib.error

PIDS = f"{LOG_DIR}/pids.json"

def stop_all():
    try: pids = json.load(open(PIDS))
    except Exception: pids = {}
    for name, pid in pids.items():
        try: os.killpg(pid, signal.SIGTERM)
        except Exception: pass
    time.sleep(1)

def start(name, cmd, cwd=None, env=None):
    log = open(f"{LOG_DIR}/{name}.log", "w")
    p = subprocess.Popen(cmd, cwd=cwd, env=env, stdout=log, stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL,
                         start_new_session=True)
    try: pids = json.load(open(PIDS))
    except Exception: pids = {}
    pids[name] = p.pid; json.dump(pids, open(PIDS, "w"))
    return p

def wait_http(url, secs=60):
    end = time.time() + secs
    while time.time() < end:
        try:
            urllib.request.urlopen(url, timeout=3); return True
        except urllib.error.HTTPError: return True
        except Exception: time.sleep(1)
    return False

stop_all()

# Lumenis reactor: serves /desktop (one2lvos-desktop.html), /lumenis-chat, /ws and /api.
# LUMENIS_EXEC_CHROOT makes its own terminal tools run inside Ubuntu 26 too.
renv = os.environ.copy()
renv.update(LUMENIS_EXEC_CHROOT=UBU_ROOT, PYTHONUNBUFFERED="1")
start("reactor", [sys.executable, "-m", "uvicorn", "lumenis-main:app", "--host", "127.0.0.1",
                  "--port", str(REACTOR_PORT)], cwd=REPO_DIR, env=renv)

# ttyd: a real terminal over a WebSocket, one Ubuntu 26 login shell per browser tab.
tenv = os.environ.copy()
tenv.pop("O2_DESKTOP_PASSWORD", None); tenv.pop("NVIDIA_API_KEY", None)
tenv.update(O2_ROOT=UBU_ROOT, O2_USER="one2lv")
start("ttyd", [TTYD, "-i", "127.0.0.1", "-p", str(TTYD_PORT), "-b", "/terminal", "-W",
               "-t", "fontSize=15", "-t", "titleFixed=Ubuntu 26 · One2lvOS", "-t", "disableLeaveAlert=true",
               "-t", 'theme={"background":"#05020c","foreground":"#d7e3ff","cursor":"#00f0ff"}',
               "bash", f"{REPO_DIR}/colab/enter-ubuntu26.sh"], env=tenv)

# Gateway: password login in front of everything, then routes /terminal to ttyd and the rest to the reactor.
genv = os.environ.copy()
genv.update(O2_GATEWAY_PASSWORD=DESKTOP_PASSWORD, O2_GATEWAY_PORT=str(GATEWAY_PORT),
            O2_REACTOR_URL=f"http://127.0.0.1:{REACTOR_PORT}", O2_TTYD_URL=f"http://127.0.0.1:{TTYD_PORT}")
genv.pop("NVIDIA_API_KEY", None)
start("gateway", [sys.executable, f"{REPO_DIR}/colab/gateway.py"], env=genv)

for name, url in (("reactor", f"http://127.0.0.1:{REACTOR_PORT}/desktop"),
                  ("ttyd", f"http://127.0.0.1:{TTYD_PORT}/terminal/"),
                  ("gateway", f"http://127.0.0.1:{GATEWAY_PORT}/_gateway/health")):
    ok = wait_http(url, 90 if name == "reactor" else 30)
    print(f"{name:8s} {'up' if ok else 'NOT RESPONDING'}   (log: {LOG_DIR}/{name}.log)")
    if not ok:
        print(open(f"{LOG_DIR}/{name}.log").read()[-2000:])

In [ ]:
#@title 5 · Open it: public link (Cloudflare quick tunnel)
import re
from IPython.display import display, HTML

try: _p = json.load(open(PIDS)).get("tunnel")
except Exception: _p = None
if _p:
    try: os.killpg(_p, signal.SIGTERM)
    except Exception: pass
start("tunnel", [CLOUDFLARED, "tunnel", "--no-autoupdate", "--url", f"http://127.0.0.1:{GATEWAY_PORT}"])
PUBLIC_URL = None
for _ in range(60):
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open(f"{LOG_DIR}/tunnel.log").read())
    if m: PUBLIC_URL = m.group(0); break
    time.sleep(1)
if not PUBLIC_URL:
    print(open(f"{LOG_DIR}/tunnel.log").read()[-1500:]); raise SystemExit("The tunnel did not start.")

# The quick-tunnel hostname takes a few seconds to resolve everywhere.
for _ in range(30):
    try:
        if json.load(urllib.request.urlopen(PUBLIC_URL + "/_gateway/health", timeout=5)).get("ok"): break
    except Exception: time.sleep(2)

display(HTML(f"""<div style="font:15px system-ui;padding:14px 18px;border:1px solid #00b8c4;border-radius:10px;background:#07030f;color:#ccd6f6">
<b style="color:#00f0ff">One2lvOS is live</b><br>
<a href="{PUBLIC_URL}/desktop" target="_blank" style="color:#7df9ff;font-size:17px">{PUBLIC_URL}/desktop</a><br>
<span style="font-size:12px;color:#8892b0">Log in with your desktop password. Terminal only: {PUBLIC_URL}/terminal/ ·
Lumenis Console: {PUBLIC_URL}/lumenis-chat · The link works while this runtime is running.</span></div>"""))

In [ ]:
#@title 6 · Status and logs (run any time)
LOG_LINES = 15  #@param {type:"integer"}
for name in ("reactor", "ttyd", "gateway", "tunnel"):
    try: pid = json.load(open(PIDS)).get(name)
    except Exception: pid = None
    alive = False
    if pid:
        try: os.kill(pid, 0); alive = True
        except Exception: pass
    print(f"── {name}: {'running' if alive else 'stopped'}")
    try: print("".join(open(f"{LOG_DIR}/{name}.log").readlines()[-LOG_LINES:]))
    except FileNotFoundError: pass

In [ ]:
#@title 7 · Stop everything
stop_all()
print("Stopped the reactor, terminal, gateway and tunnel. The Ubuntu 26 root stays at", UBU_ROOT)